# Environment

In [1]:
! pip install pandas

Looking in indexes: https://pypi.org/simple, https://pypi.org/simple

[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [9]:
!pip install --upgrade transformers accelerate

Looking in indexes: https://pypi.org/simple, https://pypi.org/simple
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 79.2 kB/s eta 0:00:0000:0400:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 77.6 kB/s eta 0:00:00a 0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 2.4/3.1 MB 102.7 kB/s eta 0:00:08^C
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 2.4/3.1 MB 102.7 kB/s eta 0:00:08


## Import Library

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import pandas as pd
import json

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Model

In [2]:
! nvidia-smi

Mon Jun 23 04:32:27 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:EF:00.0 Off |                    0 |
| N/A   28C    P0             67W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
from huggingface_hub import login
import os

token = "HF_TOKEN_REDACTED"
os.environ['HF_TOKEN'] = token

login(token=token)
print("Logged in to Hugging Face!")

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful
Logged in to Hugging Face!


### Deepseek R1-dsitill

In [4]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-7B")
model = AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-7B")

Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.33s/it]


# Load Datasets

In [5]:
import pandas as pd
cloud=pd.read_excel("../../../data/splits/cloud/cloud_test.xlsx")
device=pd.read_excel("../../../data/splits/device/device_test.xlsx")

## Ocr Load 

In [6]:
# ocr json

import json

with open("../../../data/processed/ocr/cloud/ocr_extracted_cloud_test.jsonl", "r", encoding="utf-8") as file:
    cloud_test_ocr = [json.loads(line) for line in file]

with open("../../../data/processed/ocr/device/ocr_extracted_device_test.jsonl", "r", encoding="utf-8") as file:
    device_test_ocr = [json.loads(line) for line in file]


In [7]:
from typing import List, Dict, Any

def ocr_extract(data) -> List[str]:
    results: List[str] = []

    for entry in data:
        lines=[]

        for img_i, img in enumerate(entry.get("images", [])):
            lines.append(f"img{img_i}:")

            for resp in img.get("response", []):
                if isinstance(resp, dict):
                    # --- non-table text ---
                    ntt = resp.get("non_table_text")
                    if ntt:
                        lines.append("Non-table text:")
                        lines.extend(ntt if isinstance(ntt, list) else [str(ntt)])

                    # --- tables ---
                    tables = resp.get("table")
                    if tables:
                        lines.append("Tables:")
                        for t_i, tbl in enumerate(tables if isinstance(tables, list) else [tables]):
                            lines.append(f"Table {t_i}:")
                            lines.append(tbl.get("content", "") if isinstance(tbl, dict) else str(tbl))
                    continue

                # Unknown or malformed type
                lines.append(f"[WARN] Unhandled response type: {type(resp).__name__}")
                lines.append(str(resp))

        formatted = "\n".join(lines).strip()
        if formatted:
            results.append(formatted)

    return results

In [8]:
# format ocr response for prompt

cloud_test_ocr_response_format = ocr_extract(cloud_test_ocr)   
device_test_ocr_response_format = ocr_extract(device_test_ocr)

In [9]:
len(cloud_test_ocr_response_format),len(device_test_ocr_response_format)

(1568, 947)

In [14]:
cloud['images'][1423]

"['https://i.sstatic.net/AKSXF.png']"

In [15]:
cloud_test_ocr_response_format[1423]

'img0:\nTables:\nTable 0:\n+-----------------------------------+------------+-----------+-----------+\n| EC2 Instance Type                 | Software   | EC2       | Total     |\n+-----------------------------------+------------+-----------+-----------+\n| Standard Large (m1.large)         | $0.00/hr   | $0.265/hr | $0.265/hr |\n+-----------------------------------+------------+-----------+-----------+\n| Standard XL (m1.xlarge)           | $0.00/hr   | $0.53/hr  | $0.53/hr  |\n+-----------------------------------+------------+-----------+-----------+\n| High-Memory 2XL (m2.2xlarge)      | $0.00/hr   | $0.845/hr | $0.845/hr |\n+-----------------------------------+------------+-----------+-----------+\n| High-Memory 4XL (m2.4xlarge)      | $0.00/hr   | $1.69/hr  | $1.69/hr  |\n+-----------------------------------+------------+-----------+-----------+\n| M3 Extra Large (m3.xlarge)        | $0.00/hr   | $0.475/hr | $0.475/hr |\n+-----------------------------------+------------+-----------

## Cloud

In [16]:
cloud.head(2)

,Unnamed: 0.1,Unnamed: 0,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,0,33283,Some recently changed resources are not yet pu...,\r\n \r\nEvery once in a while ...,"['java', 'eclipse', 'google-app-engine', 'java...",https://stackoverflow.com/questions/49711724/s...,4,2018-04-07 20:34:12Z,1,"[""@code511788465541441, if you consider Chanse...",[{'body': '\n \nIf that happens...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png']
1,1,34179,FAILED TO INITIALIZE RUN FROM PACKAGE.txt,\r\n \r\nOur pipeline indicates...,"['azure-web-app-service', 'azure-deployment', ...",https://stackoverflow.com/questions/65255839/f...,7,2020-12-11 17:20:22Z,2,"[""thanks for the hint. i am going to verify th...","[{'body': ""\nI am almost certain that the pack...",\r\nI am almost certain that the package path ...,"['https://i.sstatic.net/VJ1po.png', 'https://i..."


## Processed

In [17]:
cloud=cloud[['title','body','selected_answer','images']]

In [18]:
cloud.head(2)

,title,body,selected_answer,images
0,Some recently changed resources are not yet pu...,\r\n \r\nEvery once in a while ...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png']
1,FAILED TO INITIALIZE RUN FROM PACKAGE.txt,\r\n \r\nOur pipeline indicates...,\r\nI am almost certain that the package path ...,"['https://i.sstatic.net/VJ1po.png', 'https://i..."


In [19]:
cloud['context'] = "title: " + cloud['title'].astype(str) + "\nbody: " + cloud['body'].astype(str).str.strip()

In [20]:
cloud_processed=pd.DataFrame(
    {
        'context':cloud['context'],
        'gt':cloud['selected_answer'],
        'img':cloud['images']
    }
)

In [21]:
cloud_processed['ocr']=cloud_test_ocr_response_format

In [22]:
cloud_processed.head(2)

,context,gt,img,ocr
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png'],img0:\nNon-table text:\n Stale resources dete...
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\r\nI am almost certain that the package path ...,"['https://i.sstatic.net/VJ1po.png', 'https://i...",img0:\nTables:\nTable 0:\n+----+--------------...


## Device

In [23]:
device.head(2)

,Unnamed: 0,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,0,What is the purpose of the INPUT chain in the ...,\r\n \r\nWhat is the purpose of...,"['linux', 'iptables', 'linux', 'iptables']",https://serverfault.com/questions/993878/what-...,1,2019-12-01 00:23:16Z,1,"['OK, also please add this to your answer so o...",[{'body': '\nIf some NAT operation needs to be...,\r\nIf some NAT operation needs to be only don...,['https://i.sstatic.net/NUh2k.png']
1,1,"Logstash JDBC input, filter event timestamp di...",\r\n \r\nI am trying to change ...,"['elasticsearch', 'logstash', 'elastic-stack',...",https://stackoverflow.com/questions/33787795/l...,0,2015-11-18 18:38:40Z,3,"['could you suggest how I could fix?', 'Your d...",[{'body': '\nOK I finally figure how to get th...,\r\nOK I finally figure how to get the @timest...,"['https://i.sstatic.net/3nAZ9.png', 'https://i..."


## Processed

In [24]:
device=device[['title','body','selected_answer','images']]

In [25]:
device.head(2)

,title,body,selected_answer,images
0,What is the purpose of the INPUT chain in the ...,\r\n \r\nWhat is the purpose of...,\r\nIf some NAT operation needs to be only don...,['https://i.sstatic.net/NUh2k.png']
1,"Logstash JDBC input, filter event timestamp di...",\r\n \r\nI am trying to change ...,\r\nOK I finally figure how to get the @timest...,"['https://i.sstatic.net/3nAZ9.png', 'https://i..."


In [26]:
device['context'] = "title: " + device['title'].astype(str) + "\nbody: " + device['body'].astype(str).str.strip()

In [27]:
device_processed=pd.DataFrame(
    {
        'context':device['context'],
        'gt':device['selected_answer'],
        'img':device['images']
    }
)

In [28]:
device_processed['ocr']=device_test_ocr_response_format

In [29]:
device_processed.head(2)

,context,gt,img,ocr
0,title: What is the purpose of the INPUT chain ...,\r\nIf some NAT operation needs to be only don...,['https://i.sstatic.net/NUh2k.png'],img0:\nNon-table text:\n CT\n ...
1,"title: Logstash JDBC input, filter event times...",\r\nOK I finally figure how to get the @timest...,"['https://i.sstatic.net/3nAZ9.png', 'https://i...",img0:\nTables:\nTable 0:\n+-------------------...


# Prompt

## Cloud

### Zero Shot

In [30]:
zero_shot = '''
You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
{}

Question:
{}

Answer:

'''

In [31]:
def generate_zero_shot_prompt(row):
    return zero_shot.format(row['ocr'],row['context'])

cloud_processed['zero-shot-question'] = cloud_processed.apply(generate_zero_shot_prompt, axis=1)


In [32]:
print(cloud_processed['zero-shot-question'][0])


You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
img0:
Non-table text:
  Stale resources detected
                                                 ×
                                                 x
     Some recently changed resources are not yet published. Continue with launch?
 ?
                                             No
                                   Yes

Question:
title: Some recently changed resources are not yet published
body: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I've tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?

Answer:




### CoT

In [33]:
cot = '''
You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
{}

Question:
{}

Lets think step by step:
Answer:
'''

In [34]:
def generate_cot_prompt(row):
    return cot.format(row['ocr'],row['context'])

cloud_processed['cot-question'] = cloud_processed.apply(generate_cot_prompt, axis=1)


In [35]:
print(cloud_processed['cot-question'][0])


You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
img0:
Non-table text:
  Stale resources detected
                                                 ×
                                                 x
     Some recently changed resources are not yet published. Continue with launch?
 ?
                                             No
                                   Yes

Question:
title: Some recently changed resources are not yet published
body: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I've tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?

Lets think step by step:
Answer:



In [36]:
cloud_processed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1568 entries, 0 to 1567
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   context             1568 non-null   object
 1   gt                  1568 non-null   object
 2   img                 1568 non-null   object
 3   ocr                 1568 non-null   object
 4   zero-shot-question  1568 non-null   object
 5   cot-question        1568 non-null   object
dtypes: object(6)
memory usage: 73.6+ KB


## Device

### Zero Shot

In [37]:
zero_shot = '''
You are an expert device assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
{}

Question:
{}

Answer:

'''

In [38]:
def generate_zero_shot_prompt(row):
    return zero_shot.format(row['ocr'],row['context'])

device_processed['zero-shot-question'] =device_processed.apply(generate_zero_shot_prompt, axis=1)

In [39]:
print(device_processed['zero-shot-question'][0])


You are an expert device assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
img0:
Non-table text:
       CT
                                                                                              iptables
                                                                                                                                                                              Transmit Packet to
Data Link Layer
outbound interface
              Tracked by ConnTrack
                                               Receive Packet from
Data Link Layer
inbound interface
                                                                                              raw table
PREROUTING chain
                                                                              Defrag
                                                                      

### CoT

In [40]:
cot = '''
You are an expert device assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
{}

Question:
{}

Lets think step by step:
Answer:
'''

In [41]:
def generate_cot_prompt(row):
    return cot.format(row['ocr'],row['context'])

device_processed['cot-question'] = device_processed.apply(generate_cot_prompt, axis=1)


In [42]:
print(device_processed['cot-question'][0])


You are an expert device assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
img0:
Non-table text:
       CT
                                                                                              iptables
                                                                                                                                                                              Transmit Packet to
Data Link Layer
outbound interface
              Tracked by ConnTrack
                                               Receive Packet from
Data Link Layer
inbound interface
                                                                                              raw table
PREROUTING chain
                                                                              Defrag
                                                                      

In [43]:
device_processed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 947 entries, 0 to 946
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   context             947 non-null    object
 1   gt                  947 non-null    object
 2   img                 947 non-null    object
 3   ocr                 947 non-null    object
 4   zero-shot-question  947 non-null    object
 5   cot-question        947 non-null    object
dtypes: object(6)
memory usage: 44.5+ KB


## Prompt Length

In [44]:
# extract prompt length
def prompt_length(text, add_special_tokens=True):
    if pd.isnull(text) or not isinstance(text, str) or not text.strip():
        return 0
    temp_inputs = tokenizer(text, return_tensors="pt", add_special_tokens=add_special_tokens)
    return temp_inputs.input_ids.shape[1]


In [45]:
# Prompt Length -> Cloud Logs
cloud_processed['zero-shot-question-length'] = cloud_processed['zero-shot-question'].apply(prompt_length)
cloud_processed['cot-question-length'] = cloud_processed['cot-question'].apply(prompt_length)

In [46]:
# Prompt Length -> Device Logs
device_processed['zero-shot-question-length'] = device_processed['zero-shot-question'].apply(prompt_length)
device_processed['cot-question-length'] = device_processed['cot-question'].apply(prompt_length)

# Generate Response

In [47]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

### Zero-Shot

In [48]:
import os
import torch
import gc
import traceback

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
torch.set_grad_enabled(False)

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

def generate_zero_shot_answer(row):
    try:
        # Encode the question and move to the model’s device
        input_ids = tokenizer(
            row["zero-shot-question"], return_tensors="pt"
        ).input_ids.to(model.device)
        prompt_length=input_ids.shape[1]

        # Generate a completion
        output = model.generate(
            input_ids,
            max_new_tokens=768,
            do_sample=False,
            use_cache=True
        )

        # Decode and return the answer only
        return tokenizer.decode(output[0][prompt_length:], skip_special_tokens=True)

    except Exception as e:
        # Return the error string (no prompt info)
        return f"[ERROR] {e}\n{traceback.format_exc()}"

    finally:
        # Clean up GPU / RAM
        for var in ("input_ids", "output"):
            if var in locals():
                del locals()[var]
        torch.cuda.empty_cache()
        gc.collect()

### CoT

In [49]:
import os
import torch
import gc
import traceback

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
torch.set_grad_enabled(False)

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

def generate_cot_answer(row):
    try:
        input_ids = tokenizer(
            row["cot-question"], return_tensors="pt"
        ).input_ids.to(model.device)
        prompt_length=input_ids.shape[1]

        output = model.generate(
            input_ids,
            max_new_tokens=768,
            do_sample=False,
            use_cache=True,
        )

        return tokenizer.decode(output[0][prompt_length:], skip_special_tokens=True)

    except Exception as e:
        return f"[ERROR] {e}\n{traceback.format_exc()}"

    finally:
        for var in ("input_ids", "output"):
            if var in locals():
                del locals()[var]
        torch.cuda.empty_cache()
        gc.collect()


## Testing Samples

In [50]:
sample = cloud_processed.sort_values('zero-shot-question-length', ascending=False).head(3).reset_index(drop=True)

In [51]:
sample =cloud_processed[0:3]

In [52]:
sample.head(2)

,context,gt,img,ocr,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png'],img0:\nNon-table text:\n Stale resources dete...,\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,169,175
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\r\nI am almost certain that the package path ...,"['https://i.sstatic.net/VJ1po.png', 'https://i...",img0:\nTables:\nTable 0:\n+----+--------------...,\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,3224,3230


In [53]:
print(sample['zero-shot-question'][0])


You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
img0:
Non-table text:
  Stale resources detected
                                                 ×
                                                 x
     Some recently changed resources are not yet published. Continue with launch?
 ?
                                             No
                                   Yes

Question:
title: Some recently changed resources are not yet published
body: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I've tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?

Answer:




In [54]:
zero_results = []
failed_responses = []
jsonl_records = []

output_file="llm_ocr_sample_deepseek_r1_distill_qwen_7b_zero_shot_results.jsonl"
failed_file="llm_ocr_sample_deepseek_r1_distill_qwen_7b_zero_shot_failed.jsonl"

# Clear output files first
with open(output_file, "w"): pass
with open(failed_file, "w"): pass

for idx, row in tqdm(sample.iterrows(), total=len(sample)):
    answer = generate_zero_shot_answer(row)

    # Track response
    zero_results.append(answer)

    # Sample print preview
    if idx == 0:
        question_text = row['zero-shot-question']
        question_tokens = tokenizer.tokenize(question_text)[:45]
        short_question = tokenizer.convert_tokens_to_string(question_tokens)

        print("\n\nSample Question:\n")
        print(short_question)

        print("\n\nSample Response:\n")
        print(answer)

    # JSONL record
    record = {
        "row": idx,
        "question": row["zero-shot-question"],
        "response": answer
    }

    jsonl_records.append(record)
    
    with open(output_file, "a") as f:
        f.write(json.dumps(record) + "\n")

    # Track failures
    if isinstance(answer, str) and answer.startswith("[ERROR]"):
        failed_responses.append(record)
        with open(failed_file, "a") as f:
             f.write(json.dumps(failed_responses) + "\n")
    

# Add generated answers to DataFrame
sample["zero-shot-generated-answer"] = zero_results

  0%|          | 0/3 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token.As 



Sample Question:


You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text


Sample Response:

</think>

The error message indicates that some recently changed resources are not yet published. To resolve this issue, ensure that the latest code is published before starting the GAE server. If the problem persists, consider checking the project settings, restarting the server, or seeking further assistance from the Google Cloud Platform community.


 67%|██████▋   | 2/3 [00:07<00:03,  3.85s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
100%|██████████| 3/3 [00:09<00:00,  3.08s/it]
/tmp/ipykernel_117/2873790280.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample["zero-shot-generated-answer"] = zero_results


In [55]:
print(zero_results[2])

</think>

To add a "Deploy to Heroku" button that requires environment variables, you can include these variables in your app.json schema. This allows users to set the necessary variables before deployment. For example, you can add a section in your app.json like:

```json
{
  "env": {
    "HEROKU_API_KEY": "your_api_key",
    "REPOSITORY_NAME": "your_repository_name"
  }
}
```

You can then instruct users to replace the placeholder values in your README file with their actual Heroku credentials. This approach keeps the deployment process user-friendly while ensuring security.


In [56]:
import json
from tqdm import tqdm

cot_results = []
failed_cot_responses = []
cot_jsonl_records = []

output_file="llm_ocr_sample_deepseek_r1_distill_qwen_7b_cot_results.jsonl"
failed_file="llm_ocr_sample_deepseek_r1_distill_qwen_7b_cot_failed.jsonl"

# Clear output files first
with open(output_file, "w"): pass
with open(failed_file, "w"): pass


for idx, row in tqdm(sample.iterrows(), total=len(sample)):
    answer = generate_cot_answer(row)

    # Append result
    cot_results.append(answer)

    # Show sample only for first entry
    if idx == 0:
        question_text = row['cot-question']
        question_tokens = tokenizer.tokenize(question_text)[:45]
        short_question = tokenizer.convert_tokens_to_string(question_tokens)

        print("\n\nSample CoT Question:\n")
        print(short_question)

        print("\n\nSample CoT Response:\n")
        print(answer)

    # Record for JSONL
    record = {
        "index": idx,
        "question": row["cot-question"],
        "response": answer
    }
    jsonl_records.append(record)
    
    with open(output_file, "a") as f:
        f.write(json.dumps(record) + "\n")

    # Track failures
    if isinstance(answer, str) and answer.startswith("[ERROR]"):
        failed_responses.append(record)
        with open(failed_file, "a") as f:
            f.write(json.dumps(record) + "\n")
    


# Add generated CoT answers to DataFrame
sample["cot-generated-answer"] = cot_results

  0%|          | 0/3 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
 33%|███▎      | 1/3 [00:01<00:02,  1.03s/it]The attention mask and the pad token id were not set. As a co



Sample CoT Question:


You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text


Sample CoT Response:

</think>

The error message indicates that some recently changed resources are not yet published. To resolve this, you should click "Yes" to proceed with the launch. If the issue persists, consider updating your codebase or seeking further assistance from the relevant community or support channels.


 67%|██████▋   | 2/3 [00:04<00:02,  2.65s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
100%|██████████| 3/3 [00:07<00:00,  2.58s/it]
/tmp/ipykernel_117/3151197572.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample["cot-generated-answer"] = cot_results


In [57]:
print(cot_results[2])

</think>

To add a "Deploy to Heroku" button that requires environment variables, you can:

1. **Prompt Users in App.json Schema**: Include a form in your app.json that asks users to input their Heroku credentials before deploying. This allows you to capture the necessary information directly within your app.

2. **Use Heroku's Secret Key**: Generate a secret key for your Heroku app and store it in your app.json. This ensures that the credentials are securely stored and not exposed in the code.

3. **Set Defaults in Code**: If you prefer not to use a form, you can set default values for the required environment variables in your code. However, this means users will need to manually update these values in their `.env` file.

By implementing one of these methods, you can streamline the deployment process for users while maintaining security and ease of use.


In [ ]:
sample.to_csv("sample_llm_ocr_deepseek_r1_distill_qwen_7b.csv")

## Cloud Testset

In [79]:
cloud_testset=cloud_processed

In [80]:
cloud_testset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1568 entries, 0 to 1567
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   context                    1568 non-null   object
 1   gt                         1568 non-null   object
 2   img                        1568 non-null   object
 3   ocr                        1568 non-null   object
 4   zero-shot-question         1568 non-null   object
 5   cot-question               1568 non-null   object
 6   zero-shot-question-length  1568 non-null   int64 
 7   cot-question-length        1568 non-null   int64 
dtypes: int64(2), object(6)
memory usage: 98.1+ KB


In [81]:
cloud_testset.head(2)

,context,gt,img,ocr,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png'],img0:\nNon-table text:\n Stale resources dete...,\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,169,175
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\r\nI am almost certain that the package path ...,"['https://i.sstatic.net/VJ1po.png', 'https://i...",img0:\nTables:\nTable 0:\n+----+--------------...,\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,3224,3230


### zero shot response

In [82]:
import json
from tqdm import tqdm

zero_cloud_results = []
zero_cloud_jsonl_records = []
zero_cloud_failed_responses = []

output_file="llm_ocr_cloud_deepseek_r1_distill_qwen_7b_zero_shot_results.jsonl"
failed_file="llm_ocr_cloud_deepseek_r1_distill_qwen_7b_zero_shot_failed.jsonl"

# Clear output files first
with open(output_file, "w"): pass
with open(failed_file, "w"): pass

for idx, row in tqdm(cloud_testset.iterrows(), total=len(cloud_testset)):
    # Generate answer
    answer = generate_zero_shot_answer(row)
    zero_cloud_results.append(answer)

    # Show the first response as a sample
    if idx == 0:
        question_text = row['zero-shot-question']
        question_tokens = tokenizer.tokenize(question_text)[:45]
        short_question = tokenizer.convert_tokens_to_string(question_tokens)

        print("\n\nSample Cloud Zero-Shot Question:\n")
        print(short_question)

        print("\n\nSample Cloud Zero-Shot Response:\n")
        print(answer)

    # Prepare JSONL record
    record = {
        "row": idx,
        "question": row["zero-shot-question"],
        "response": answer
    }
    zero_cloud_jsonl_records.append(record)

    with open(output_file, "a") as f:
        f.write(json.dumps(record) + "\n")

    # Track failures
    if isinstance(answer, str) and answer.startswith("[ERROR]"):
        failed_responses.append(record)
        with open(failed_file, "a") as f:
          f.write(json.dumps(record) + "\n")

# Save results in the DataFrame
cloud_testset["zero-shot-generated-answer"] = zero_cloud_results

  0%|          | 0/1568 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
  0%|          | 1/1568 [00:01<28:54,  1.11s/it]The attention mask and the pad token id were not set. A



Sample Cloud Zero-Shot Question:


You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text


Sample Cloud Zero-Shot Response:

</think>

The error message indicates that some recently changed resources are not yet published. To resolve this issue, ensure that the latest code is published before starting the GAE server. If the problem persists, consider checking the project settings, restarting the server, or seeking further assistance from the Google Cloud Platform community.


  0%|          | 2/1568 [00:06<1:30:17,  3.46s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
  0%|          | 3/1568 [00:08<1:13:42,  2.83s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
  0%|          | 4/1568 [00:11<1:18:48,  3.02s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
  0%|          | 5/1568 [00:14<1:14:27,  2.86s/it]The attention mask and the pad token id were

In [83]:
zero_cloud_results[1]

'</think>\n\nThe issue arises because the artifact size in the deployment is incorrectly reported as 1 KB, while the actual artifact size is 17 MB. This discrepancy causes confusion about why the ZIP file cannot be extracted. The correct size is 17 MB, which aligns with the artifact observed when dragging and dropping the ZIP file into the Kudo Console. This indicates that the size reported in the deployment is incorrect, possibly due to a misconfiguration or error in the deployment task. The correct behavior is that the artifact size should match the actual size of the ZIP file, allowing it to be extracted and deployed successfully.\n\nAnswer:\n\nThe size of the artifact in the deployment is incorrectly reported as 1 KB, while the actual size of the ZIP file is 17 MB. This discrepancy causes the issue of the ZIP file not being extractable. The correct size should be 17 MB, which is the actual size of the artifact.'

In [84]:
cloud_testset.head(2)

,context,gt,img,ocr,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length,zero-shot-generated-answer
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png'],img0:\nNon-table text:\n Stale resources dete...,\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,169,175,</think>\n\nThe error message indicates that s...
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\r\nI am almost certain that the package path ...,"['https://i.sstatic.net/VJ1po.png', 'https://i...",img0:\nTables:\nTable 0:\n+----+--------------...,\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,3224,3230,</think>\n\nThe issue arises because the artif...


In [85]:
cloud_testset.to_excel("../../../outputs/responses/baselines/llm_ocr/cloud/llm_ocr_response_cloud_deepseek_r1_distill_qwen_7b_zero_shot.xlsx", index=False)

### Cot Response

In [ ]:
import json
from tqdm import tqdm

cot_cloud_results = []
failed_cloud_cot_responses = []
cot_cloud_jsonl_records = []

output_file="llm_ocr_cloud_deepseek_r1_distill_qwen_7b_cot_results.jsonl"
failed_file="llm_ocr_cloud_deepseek_r1_distill_qwen_7b_cot_failed.jsonl"

# Clear output files first
with open(output_file, "w"): pass
with open(failed_file, "w"): pass


for idx, row in tqdm(cloud_testset.iterrows(), total=len(cloud_testset)):
    # Generate answer using CoT function
    answer = generate_cot_answer(row)
    cot_cloud_results.append(answer)

    # Sample output (first only)
    if idx == 0:
        question_text = row['cot-question']
        question_tokens = tokenizer.tokenize(question_text)[:45]
        short_question = tokenizer.convert_tokens_to_string(question_tokens)

        print("\n\nSample Cloud CoT Question:\n")
        print(short_question)

        print("\n\nSample Cloud CoT Response:\n")
        print(answer)

    # Prepare JSONL entry
    record = {
        "row": idx,
        "question": row["cot-question"],
        "response": answer
    }
    cot_cloud_jsonl_records.append(record)
    with open(output_file, "a") as f:
        f.write(json.dumps(record) + "\n")

    # Track failures
    if isinstance(answer, str) and answer.startswith("[ERROR]"):
        failed_responses.append(record)
        with open(failed_file, "a") as f:
            f.write(json.dumps(record) + "\n")

# Append column to DataFrame
cloud_testset["cot-generated-answer"] = cot_cloud_results

  0%|          | 0/1568 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
  0%|          | 1/1568 [00:00<25:38,  1.02it/s]The attention mask and the pad token id were not set. A



Sample Cloud CoT Question:


You are an expert cloud assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text


Sample Cloud CoT Response:

</think>

The error message indicates that some recently changed resources are not yet published. To resolve this, you should click "Yes" to proceed with the launch. If the issue persists, consider updating your codebase or seeking further assistance from the relevant community or support channels.


  0%|          | 2/1568 [00:04<1:06:58,  2.57s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
  0%|          | 3/1568 [00:07<1:10:42,  2.71s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
  0%|          | 4/1568 [00:11<1:19:38,  3.06s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
  0%|          | 5/1568 [00:12<1:03:10,  2.43s/it]The attention mask and the pad token id were

In [ ]:
cloud_testset.head(2)

In [ ]:
cloud_testset.to_excel("../../../outputs/responses/baselines/llm_ocr/cloud/llm_ocr_response_cloud_deepseek_r1_distill_qwen_7b_zero_shot.xlsx", index=False)

## Device Testset

In [58]:
device_testset=device_processed

In [59]:
device_testset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 947 entries, 0 to 946
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   context                    947 non-null    object
 1   gt                         947 non-null    object
 2   img                        947 non-null    object
 3   ocr                        947 non-null    object
 4   zero-shot-question         947 non-null    object
 5   cot-question               947 non-null    object
 6   zero-shot-question-length  947 non-null    int64 
 7   cot-question-length        947 non-null    int64 
dtypes: int64(2), object(6)
memory usage: 59.3+ KB


In [60]:
device_testset.shape

(947, 8)

### Zero shot Response

In [61]:
import json
from tqdm import tqdm

# Lists for tracking
zero_device_results = []
zero_device_jsonl_records = []
zero_device_failed_responses = []

output_file="llm_ocr_device_deepseek_r1_distill_qwen_7b_zero_shot_results.jsonl"
failed_file="llm_ocr_device_deepseek_r1_distill_qwen_7b_zero_shot_failed.jsonl"

# Clear output files first
with open(output_file, "w"): pass
with open(failed_file, "w"): pass

for idx, row in tqdm(device_testset.iterrows(), total=len(device_testset)):
    # Generate zero-shot answer
    answer = generate_zero_shot_answer(row)
    zero_device_results.append(answer)

    # Print one sample preview
    if idx == 0:
        question_text = row["zero-shot-question"]
        question_tokens = tokenizer.tokenize(question_text)[:45]
        short_question = tokenizer.convert_tokens_to_string(question_tokens)

        print("\n\nSample Device Zero-Shot Question:\n")
        print(short_question)

        print("\n\nSample Device Zero-Shot Response:\n")
        print(answer)

    # Build JSONL record
    record = {
        "row": idx,
        "question": row["zero-shot-question"],
        "response": answer
    }
    zero_device_jsonl_records.append(record)

    with open(output_file, "a") as f:
        f.write(json.dumps(record) + "\n")

    # Track failures
    if isinstance(answer, str) and answer.startswith("[ERROR]"):
        failed_responses.append(record)
        with open(failed_file, "a") as f:
            f.write(json.dumps(record) + "\n")


# Add results to DataFrame
device_testset["zero-shot-generated-answer"] = zero_device_results

  0%|          | 0/947 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
  0%|          | 1/947 [00:00<14:10,  1.11it/s]The attention mask and the pad token id were not set. As 



Sample Device Zero-Shot Question:


You are an expert device assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text


Sample Device Zero-Shot Response:

</think>

The purpose of the INPUT chain in the nat table is to handle the translation of source addresses (SNAT) for incoming packets, allowing for source address translation which is essential for establishing connections to external networks.


  0%|          | 2/947 [00:18<2:49:55, 10.79s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
  0%|          | 3/947 [00:20<1:43:40,  6.59s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
  0%|          | 4/947 [00:24<1:26:33,  5.51s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
  1%|          | 5/947 [00:37<2:09:15,  8.23s/it]The attention mask and the pad token id were not

In [62]:
device_testset.head(2)

,context,gt,img,ocr,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length,zero-shot-generated-answer
0,title: What is the purpose of the INPUT chain ...,\r\nIf some NAT operation needs to be only don...,['https://i.sstatic.net/NUh2k.png'],img0:\nNon-table text:\n CT\n ...,\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,532,538,</think>\n\nThe purpose of the INPUT chain in ...
1,"title: Logstash JDBC input, filter event times...",\r\nOK I finally figure how to get the @timest...,"['https://i.sstatic.net/3nAZ9.png', 'https://i...",img0:\nTables:\nTable 0:\n+-------------------...,\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,4902,4908,</think>\n\nThe issue arises because the `@tim...


In [63]:
device_testset.to_excel("../../../outputs/responses/baselines/llm_ocr/device/llm_ocr_response_device_deepseek_r1_distill_qwen_7b_zero_shot.xlsx", index=False)

### CoT Response

In [64]:
import json
from tqdm import tqdm

# Containers
cot_device_results = []
cot_device_jsonl_records = []
cot_device_failed_responses = []

output_file="llm_ocr_device_deepseek_r1_distill_qwen_7b_cot_results.jsonl"
failed_file="llm_ocr_device_deepseek_r1_distill_qwen_7b_cot_failed.jsonl"

# Clear output files first
with open(output_file, "w"): pass
with open(failed_file, "w"): pass

for idx, row in tqdm(device_testset.iterrows(), total=len(device_testset)):
    # Generate CoT answer
    answer = generate_cot_answer(row)
    cot_device_results.append(answer)

    # Print one sample preview
    if idx == 0:
        question_text = row["cot-question"]
        question_tokens = tokenizer.tokenize(question_text)[:45]
        short_question = tokenizer.convert_tokens_to_string(question_tokens)

        print("\n\nSample Device CoT Question:\n")
        print(short_question)

        print("\n\nSample Device CoT Response:\n")
        print(answer)

    # JSONL record
    record = {
        "row": idx,
        "question": row["cot-question"],
        "response": answer
    }
    cot_device_jsonl_records.append(record)

    with open(output_file, "a") as f:
        f.write(json.dumps(jsonl_records) + "\n")

    # Track failures
    if isinstance(answer, str) and answer.startswith("[ERROR]"):
        failed_responses.append(record)
        with open(failed_file, "a") as f:
            f.write(json.dumps(failed_responses) + "\n")

# Write results back to DataFrame
device_testset["cot-generated-answer"] = cot_device_results

  0%|          | 0/947 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
  0%|          | 1/947 [00:01<26:45,  1.70s/it]The attention mask and the pad token id were not set. As 



Sample Device CoT Question:


You are an expert device assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text


Sample Device CoT Response:

</think>

The purpose of the INPUT chain in the nat table is to handle packets that originate from the local network, allowing them to be routed to the appropriate destination without modification. This is distinct from the PREROUTING chain, which processes incoming packets from external networks, applying DNAT for port forwarding. The INPUT chain is typically used for managing local traffic, such as handling local applications or services, whereas the PREROUTING chain is for external traffic management.


  0%|          | 2/947 [00:15<2:19:15,  8.84s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
  0%|          | 3/947 [00:17<1:27:55,  5.59s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
  0%|          | 4/947 [00:21<1:17:47,  4.95s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
  1%|          | 5/947 [00:21<52:45,  3.36s/it]  The attention mask and the pad token id were not

In [78]:
print(cot_device_results[123])

</think>

To examine your Windows Firewall outbound rules, use the **Firewall** command in the Command Prompt. Here's how:

1. Open the Command Prompt.
2. Type `firewall.exe` and press Enter.
3. In the Firewall Configuration window, click **View** > **Outbound Rules**.
4. Review the outbound rules, including the one for port 9000.

If the rule isn't listed, check the **General** menu for any additional outbound rules. If the issue persists, ensure the program is correctly listening on port 9000 and that the firewall permissions are set properly.


In [65]:
device_testset.head(2)

,context,gt,img,ocr,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length,zero-shot-generated-answer,cot-generated-answer
0,title: What is the purpose of the INPUT chain ...,\r\nIf some NAT operation needs to be only don...,['https://i.sstatic.net/NUh2k.png'],img0:\nNon-table text:\n CT\n ...,\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,532,538,</think>\n\nThe purpose of the INPUT chain in ...,</think>\n\nThe purpose of the INPUT chain in ...
1,"title: Logstash JDBC input, filter event times...",\r\nOK I finally figure how to get the @timest...,"['https://i.sstatic.net/3nAZ9.png', 'https://i...",img0:\nTables:\nTable 0:\n+-------------------...,\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,4902,4908,</think>\n\nThe issue arises because the `@tim...,</think>\n\nThe user is experiencing a discrep...


In [66]:
device_testset.to_excel("../../../outputs/responses/baselines/llm_ocr/device/llm_ocr_response_device_deepseek_r1_distill_qwen_7b_zero_shot.xlsx", index=False)